# sindyc_08: how messy are the derivatives, and is SINDy viable on them?

SINDy regresses derivatives on a library. In SMAClite health drops in whole shots, units die one
at a time and reach changes one unit at a time, so the "derivative" is a sawtooth of spikes and
zeros. This notebook looks at that directly:

1. what the raw and smoothed derivatives look like in a battle;
2. how step-like each state is, and whether the steps come in synchronised volleys;
3. how much of the raw derivative is signal at the time scale the model works on;
4. whether the fitted equations depend on how the derivative is estimated (smoothing window,
   weak form), and whether that matters for predictions.

Data: exp06 validation battles for the plots, training battles for the fits (exploratory data).

In [ ]:
import pickle
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

sys.path.insert(0, "../lib")  # helpers: notebooks/lib
import sindyc_07_lab as lab  # noqa: E402
from weak_sindy import fit_weak, weak_rows  # noqa: E402

from sindy_wm.models.intent_model import DT, IntentModel, add_controls, add_rates, r2  # noqa: E402
from sindy_wm.paths import DATA_DIR  # noqa: E402

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
PROCESSED = DATA_DIR / "processed" / "exp06_intents"
current = pickle.load(open(PROCESSED / "model_derived.pkl", "rb"))
val = add_controls(pd.read_parquet(PROCESSED / "val.parquet"))
train = add_controls(pd.read_parquet(PROCESSED / "train.parquet"))
SHOT = 6 / 450  # one marine shot on the common scale (health 1 = 450 hp)
COOLDOWN = 0.61  # seconds between shots of one marine
COLS = ["health_blue", "health_red", "alive_blue", "alive_red", "distance", "reach_blue", "reach_red"]
both = (val["alive_blue"] > 0) & (val["alive_red"] > 0)
contact_t = val["t"].where(val["reach_blue"] + val["reach_red"] > 0).groupby(level=0).transform("min")
val["since_contact"] = val["t"] - contact_t
print(
    f"{val.index.get_level_values(0).nunique()} validation battles, dt = {DT} s; one shot = {SHOT:.4f}; "
    f"cooldown {COOLDOWN} s = {COOLDOWN / DT:.2f} steps"
)

## 1 One battle: the states and three estimates of their derivative

Raw = forward difference per step (Δx / Δt). SG 9 = Savitzky-Golay over 9 steps (1.1 s), what the
model was fitted on. SG 33 = over 33 steps (4.1 s).

In [ ]:
info = val.groupby(level=0).agg(seconds=("t", "max"), won=("battle_won", "first"))
ep = info[info["seconds"].between(10, 12)].index[0]
g = val.loc[ep]
t = g["t"].to_numpy()


def raw_rate(x):
    return np.r_[np.diff(x) / DT, np.nan]


fig, axes = plt.subplots(4, 2, figsize=(15, 11), sharex=True)
for row, (label, cols, f) in enumerate(
    [
        ("health", ["health_blue", "health_red"], 1),
        ("units alive", ["alive_blue", "alive_red"], 10),
        ("distance", ["distance"], 1),
        ("units in reach", ["reach_blue", "reach_red"], 10),
    ]
):
    for c, color in zip(cols, ("tab:blue", "tab:red"), strict=False):
        x = g[c].to_numpy() * f
        axes[row, 0].plot(t, x, color=color, drawstyle="steps-post", label=c)
        axes[row, 1].plot(
            t, raw_rate(x), color=color, alpha=0.35, lw=0.8, label=f"{c}: raw" if row == 0 else None
        )
        for w, ls in ((9, "-"), (33, "--")):
            axes[row, 1].plot(
                t,
                savgol_filter(x, w, 2, deriv=1, delta=DT),
                color=color,
                ls=ls,
                lw=1.6,
                label=f"{c}: SG {w}" if row == 0 else None,
            )
    axes[row, 0].set(ylabel=label)
    axes[row, 1].set(ylabel=f"d({label})/dt")
    axes[row, 1].axhline(0, color="gray", lw=0.5)
axes[0, 0].legend(fontsize=8)
axes[0, 1].legend(fontsize=7, ncol=2)
axes[-1, 0].set(xlabel="seconds")
axes[-1, 1].set(xlabel="seconds")
fig.suptitle(f"battle {ep}: states (left) and derivatives (right)")
fig.tight_layout()

Zoom on the first 4 s of contact: health per step in units of one shot.

In [ ]:
c0 = g["since_contact"].to_numpy()
sel = (c0 >= -0.5) & (c0 <= 4)
fig, ax = plt.subplots(1, 2, figsize=(15, 3.3))
for c, color in (("health_blue", "tab:blue"), ("health_red", "tab:red")):
    shots = -np.diff(g[c].to_numpy()) / SHOT
    ax[0].bar(
        c0[:-1][sel[:-1]] + (0.03 if c == "health_red" else -0.03),
        shots[sel[:-1]],
        width=0.06,
        color=color,
        label=f"{c}: shots taken per step",
    )
    x = g[c].to_numpy()
    ax[1].plot(c0[sel], raw_rate(x)[sel], color=color, alpha=0.4, drawstyle="steps-post")
    ax[1].plot(c0[sel], savgol_filter(x, 9, 2, deriv=1, delta=DT)[sel], color=color, lw=2)
ax[0].set(xlabel="seconds since first contact", ylabel="shots (6 hp each)")
ax[0].legend(fontsize=8)
ax[1].set(xlabel="seconds since first contact", ylabel="health rate: raw (thin), SG 9 (thick)")
fig.tight_layout()

## 2 How step-like is each state?

Per step, while both sides are alive.

In [ ]:
d = val.groupby(level=0)[COLS].diff()[both]
quant = {
    "health_blue": SHOT,
    "health_red": SHOT,
    "alive_blue": 0.1,
    "alive_red": 0.1,
    "reach_blue": 0.1,
    "reach_red": 0.1,
    "distance": np.nan,
}
step_table = pd.DataFrame(
    {
        c: {
            "steps with no change (%)": 100 * (d[c].abs() < 1e-9).mean(),
            "smallest possible change": quant[c],
            "median |change| when it changes": d.loc[d[c].abs() > 1e-9, c].abs().median(),
        }
        for c in COLS
    }
).T
display(step_table.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 3.3))
h = -d["health_blue"].dropna() / SHOT
axes[0].hist(h[h.abs() < 12], bins=np.arange(-0.5, 12.5, 1), color="tab:blue", alpha=0.7)
axes[0].set(xlabel="Blue health lost in one step, in shots", ylabel="steps", yscale="log")
a = -10 * d["alive_red"].dropna()
axes[1].hist(a, bins=np.arange(-0.5, 5.5, 1), color="tab:red", alpha=0.7)
axes[1].set(xlabel="Red units dying in one step", ylabel="steps", yscale="log")
fig.tight_layout()

## 3 Is the sawtooth noise, or a rhythm? Volleys and the spectrum

A marine fires every 0.61 s. If many units start firing at the same moment (first contact), a
team's damage comes in synchronised volleys, which shows up as a periodic signal at
1 / 0.61 = 1.64 Hz. The sampling rate is 8 Hz (Nyquist 4 Hz), so 1.64 Hz is resolved; 0.61 s is
4.9 steps, not a whole number, so volleys land on alternating 5- and 4-step gaps.

Below: the autocorrelation of each team's health change per step, early in contact (0-3 s) and
later (3-8 s), and the average power spectrum of the raw health rate during contact.

In [ ]:
inc = val.groupby(level=0)["health_red"].diff().where(both)  # Red's loss = Blue's fire
early = val["since_contact"].between(0, 3)
late = val["since_contact"].between(3, 8)
lags = np.arange(1, 17)
auto = {}
for label, m in (("0-3 s after contact", early), ("3-8 s after contact", late)):
    x = inc.where(m)
    by_ep = x.groupby(level=0)
    auto[label] = [x.corr(by_ep.shift(-k)) for k in lags]
auto = pd.DataFrame(auto, index=lags * DT)
auto.index.name = "lag (s)"

spec = []
for _, gg in val[both].groupby(level=0):
    for c in ("health_blue", "health_red"):
        x = np.diff(gg[c].to_numpy()) / DT
        if len(x) >= 32:
            seg = x[:32] - x[:32].mean()
            spec.append(np.abs(np.fft.rfft(seg * np.hanning(32))) ** 2)
freq = np.fft.rfftfreq(32, DT)
spec = np.mean(spec, axis=0)

fig, ax = plt.subplots(1, 2, figsize=(15, 3.6))
for col in auto:
    ax[0].plot(auto.index, auto[col], "o-", label=col)
ax[0].axhline(0, color="gray", lw=0.5)
for k in range(1, 4):
    ax[0].axvline(k * COOLDOWN, color="gray", ls=":", lw=1)
ax[0].set(
    xlabel="lag (s); dotted: multiples of the 0.61 s cooldown",
    ylabel="autocorrelation of Red's health change per step",
)
ax[0].legend()
ax[1].plot(freq, spec / spec.sum(), "o-")
for k in (1, 2):
    ax[1].axvline(k / COOLDOWN, color="gray", ls=":")
ax[1].set(
    xlabel="frequency (Hz); dotted: 1 and 2 × 1/0.61 s",
    ylabel="share of power",
    title="raw health rate, first 4 s of contact (both sides)",
)
fig.tight_layout()
auto.T.round(2)

## 4 How much of the raw derivative is signal?

Three numbers per state, during contact:

- the share of the raw rate's variance below 1 Hz (slower than one volley; the time scale the
  team-level model can describe);
- the R² between the raw rate and the SG 9 rate (how much of the raw rate the smoothing keeps);
- for health: the R² of the model's own prediction against the raw rate and against the SG 9 rate
  (with measured reach). If the model explains the smoothed rate well but the raw rate poorly,
  the difference is shot-timing, not model error.

In [ ]:
def lowpass_share(x, cutoff=1.0):
    """Share of the variance of x (one battle's raw rate) below `cutoff` Hz."""
    x = x - x.mean()
    F = np.fft.rfft(x)
    f = np.fft.rfftfreq(len(x), DT)
    p = np.abs(F) ** 2
    return p[f < cutoff].sum() / p.sum()


sig = []
raw_all, sg_all = {c: [] for c in COLS}, {c: [] for c in COLS}
pred_all = {"health_blue": [], "health_red": []}
for _, gg in val[both].groupby(level=0):
    if len(gg) < 24:
        continue
    for c in COLS:
        x = gg[c].to_numpy()
        raw_all[c].append(np.gradient(x, DT))
        sg_all[c].append(savgol_filter(x, 9, 2, deriv=1, delta=DT))
    pred_all["health_blue"].append(-0.177 * gg["reach_red"].to_numpy())
    pred_all["health_red"].append(
        (-0.1861 + 0.1633 * gg["withdraw"].to_numpy()) * gg["reach_blue"].to_numpy()
    )
for c in COLS:
    raw, sg = np.concatenate(raw_all[c]), np.concatenate(sg_all[c])
    row = {
        "state": c,
        "variance below 1 Hz (%)": 100 * np.mean([lowpass_share(x) for x in raw_all[c] if x.std() > 0]),
        "R²(SG 9 vs raw)": r2(raw, sg),
    }
    if c in pred_all:
        p = np.concatenate(pred_all[c])
        row |= {"model R² vs raw rate": r2(raw, p), "model R² vs SG 9 rate": r2(sg, p)}
    sig.append(row)
pd.DataFrame(sig).set_index("state").round(3)

## 5 Do the equations depend on how the derivative is estimated?

`model_derived`'s fit (measured reach, health threshold 0.15) repeated with Savitzky-Golay windows
of 5, 9 (as fitted), 17 and 33 steps, and in weak form (integrals, no derivative) with windows of
0.5, 1 and 2 s. Same library and thresholds throughout. Then each version rolled out on the
validation battles.

In [ ]:
def coef_row(m):
    out = {}
    for eq in ("health_blue", "health_red", "distance"):
        out |= {f"{eq}: {n}": c for n, c in zip(m.names, m.coef[eq], strict=True) if c}
    for s in ("blue", "red"):
        out |= {f"alive {s}: {k}": v for k, v in m.p[s].items() if v}
    out |= {
        f"fit R² {k}": v
        for k, v in m.fit_r2.items()
        if k in ("health_blue", "health_red", "distance", "alive_blue", "alive_red")
    }
    return pd.Series(out)


Bv = lab.load_batch("val")
fits, scores = {}, {}
both_tr = (train["alive_blue"] > 0) & (train["alive_red"] > 0)
for w in (5, 9, 17, 33):
    rows = add_rates(
        train, cols=("health_blue", "health_red", "alive_blue", "alive_red", "distance"), window=w
    )
    rows = rows[(rows["alive_blue"] > 0) & (rows["alive_red"] > 0)]
    m = IntentModel(current.P, exclude=current.exclude).fit(
        rows, reach_true=True, thr_health=0.15, thr_shape=None
    )
    fits[f"SG {w} ({w * DT:.2f} s)"] = m
for radius in (4, 8, 16):
    fits[f"weak {radius * DT:g} s"] = fit_weak(
        current, weak_rows(train, current.P, current.names, radius=radius)
    )
for k, m in fits.items():
    scores[k] = lab.metrics(m.simulate(Bv["X0"], Bv["shape0"], Bv["U"], reach0=Bv["reach0"]), Bv)
coef_table = pd.DataFrame({k: coef_row(m) for k, m in fits.items()})
display(coef_table.round(3))
S = pd.DataFrame(scores).T
S[[c for c in S.columns if c.startswith(("health", "margin"))]].round(3)

## 6 Where the rhythm comes from: cooldown in game ticks

SMAClite counts cooldown down in game ticks of 1/16 s (`unit.py`: `cooldown -= GAME_TICK_TIME`),
and a unit can only fire on a tick. A marine's 0.61 s cooldown therefore lasts 10 ticks =
**0.625 s = exactly 5 steps** of the data (2 ticks per step). Every marine fires on a fixed 5-step
beat, and units that start together stay in phase. One marine's cooldown in the snapshots:

In [ ]:
from sindy_wm.io import storage  # noqa: E402
from sindy_wm.paths import RAW_DIR  # noqa: E402

snap = storage.load_snapshots(RAW_DIR / "exp06_intents" / "val")
snap = snap[snap["alive"]]
one = snap[(snap["episode_idx"] == 3) & (snap["team"] == "blue") & (snap["unit_id"] == 0)].set_index("step")[
    ["cooldown", "target_in_range", "hp"]
]
display(one.loc[28:46].T.round(3))
EFFECTIVE = 10 / 16
print(
    f"effective cooldown {EFFECTIVE} s -> damage per second {6 / EFFECTIVE:.2f} "
    f"(not 6 / 0.61 = {6 / 0.61:.2f}); "
    f"health rule on the common scale {10 * 6 / EFFECTIVE / 450:.3f} (not 0.219)"
)

## 7 Smoothing over whole volley periods

A centred moving average over exactly 5 steps removes the 5-step rhythm and its harmonics
completely (a comb filter); SG 9 does not, because 9 is not a multiple of 5. Rates compared:

- SG 9 (as fitted today);
- box 5 then central difference; box 5 applied twice (a 9-step triangle, still zero at the
  rhythm) then central difference; box 15 then central difference;
- box 5 then SG 9.

For each: how much rhythm is left in the health rate (power at 1.6 Hz and 3.2 Hz), how well the
health equation explains the rate, the refitted coefficients, and the rollout on validation.

In [ ]:
from scipy.ndimage import uniform_filter1d  # noqa: E402

RATE_COLS = ("health_blue", "health_red", "alive_blue", "alive_red", "distance")


def box(x, k, times=1):
    for _ in range(times):
        x = uniform_filter1d(x, k, mode="nearest")
    return x


METHODS = {
    "SG 9": lambda x: savgol_filter(x, 9, 2, deriv=1, delta=DT),
    "box 5": lambda x: np.gradient(box(x, 5), DT),
    "box 5 twice": lambda x: np.gradient(box(x, 5, 2), DT),
    "box 15": lambda x: np.gradient(box(x, 15), DT),
    "box 5 + SG 9": lambda x: savgol_filter(box(x, 5), 9, 2, deriv=1, delta=DT),
}


def with_rates(df, method):
    out = []
    for _, gg in df.groupby(level=0, sort=False):
        if len(gg) <= 15:
            continue
        gg = gg.copy()
        for c in RATE_COLS:
            gg["d_" + c] = METHODS[method](gg[c].to_numpy(float))
        out.append(gg)
    return pd.concat(out)


def rhythm_share(df):
    """Share of the health rate's power at the volley frequency and its harmonic, first 4 s of contact."""
    p = []
    for _, gg in df[(df["alive_blue"] > 0) & (df["alive_red"] > 0)].groupby(level=0):
        for c in ("d_health_blue", "d_health_red"):
            x = gg[c].to_numpy()[:32]
            if len(x) == 32 and x.std() > 0:
                p.append(np.abs(np.fft.rfft((x - x.mean()) * np.hanning(32))) ** 2)
    p = np.mean(p, axis=0)
    f = np.fft.rfftfreq(32, DT)
    return p[(np.abs(f - 1.6) < 0.2) | (np.abs(f - 3.2) < 0.2)].sum() / p.sum()


vol_fit, vol_roll, vol_info = {}, {}, {}
for method in METHODS:
    tr_r = with_rates(train, method)
    tr_r = tr_r[(tr_r["alive_blue"] > 0) & (tr_r["alive_red"] > 0)]
    va_r = with_rates(val, method)
    m = IntentModel(current.P, exclude=current.exclude).fit(
        tr_r, reach_true=True, thr_health=0.15, thr_shape=None
    )
    vol_fit[method] = coef_row(m)
    vol_roll[method] = lab.metrics(m.simulate(Bv["X0"], Bv["shape0"], Bv["U"], reach0=Bv["reach0"]), Bv)
    vb = va_r[(va_r["alive_blue"] > 0) & (va_r["alive_red"] > 0)]
    vol_info[method] = {
        "rhythm power left (%)": 100 * rhythm_share(va_r),
        "health eq. R² on validation (Blue)": r2(vb["d_health_blue"], -0.177 * vb["reach_red"]),
        "health eq. R² on validation (Red)": r2(
            vb["d_health_red"], (-0.1861 + 0.1633 * vb["withdraw"]) * vb["reach_blue"]
        ),
    }
display(pd.DataFrame(vol_info).T.round(3))
display(pd.DataFrame(vol_fit).round(3))
V = pd.DataFrame(vol_roll).T
V[[c for c in V.columns if c.startswith(("health", "margin"))]].round(3)

## 8 Would cooldown help as a state?

**8a Per step: can cooldown explain the sawtooth?** A unit fires during the next step if its
target is in range and its cooldown is at most 0.0475 s (the last value before it fires, see
section 6). Count those per side and compare with the enemy's health lost in the next step,
against the count of units in reach (what the model uses).

In [ ]:
snap["fires_next"] = snap["target_in_range"].astype(bool) & (snap["cooldown"] <= 0.05)
fires = snap.groupby(["episode_idx", "step", "team"])["fires_next"].sum().unstack("team").fillna(0)
nxt = val[COLS].join(fires, on=["episode_idx", "step"])
nxt[["blue", "red"]] = nxt[["blue", "red"]].fillna(0)
lost_next = -val.groupby(level=0)[["health_blue", "health_red"]].diff().groupby(level=0).shift(-1) / SHOT
ok8 = both & lost_next.notna().all(axis=1)
per_step = {}
for side, enemy in (("blue", "red"), ("red", "blue")):
    y = lost_next.loc[ok8, f"health_{side}"].to_numpy()
    for label, x in (
        (f"{enemy} units in reach", 10 * nxt.loc[ok8, f"reach_{enemy}"].to_numpy()),
        (f"{enemy} units about to fire (cooldown)", nxt.loc[ok8, enemy].to_numpy()),
    ):
        k = (x @ y) / (x @ x)
        per_step[(f"{side} health lost next step (shots)", label)] = {"shots per unit": k, "R²": r2(y, k * x)}
display(pd.DataFrame(per_step).T.round(3))

**8b Over a horizon: how much error can volley timing cause at most?** Over h seconds a side
receives a whole number of volleys; a smooth-rate model spreads them out. The timing-only error
is the difference between the real health change over h and the same change measured on the
5-step-averaged health (rhythm removed), from the same start points as the rollouts (every 8 steps).
This is roughly the most a cooldown or volley-phase state could remove from the health error.

In [ ]:
timing = {}
Hraw = val[["health_blue", "health_red"]]
Hbox = Hraw.groupby(level=0).transform(lambda x: box(x.to_numpy(float), 5))
for h in (1, 2, 4, 8):
    n = round(h / DT)
    d_raw = Hraw.groupby(level=0).shift(-n) - Hraw
    d_box = Hbox.groupby(level=0).shift(-n) - Hbox
    start = (
        (val.groupby(level=0).cumcount() % 8 == 0)
        & (val.groupby(level=0).cumcount() >= 8)
        & d_raw.notna().all(axis=1)
    )
    timing[f"{h} s"] = {
        "timing-only error (health)": (d_raw[start] - d_box[start]).abs().to_numpy().mean(),
        "model_derived health error": lab.metrics(lab.simulate(lab.base_params(), Bv), Bv)[f"health {h}s"],
    }
timing = pd.DataFrame(timing).T
timing["timing share of the model's error (%)"] = 100 * timing.iloc[:, 0] / timing.iloc[:, 1]
timing.round(3)

## 9 Does the model get better or worse in larger battles? (mean-field check)

exp06 battles range from 11 to 33 units in total. If describing a team by totals is a good
approximation, errors *relative to the size of the battle* should not grow with size (and may
shrink, as averages over more units are smoother). Per size group, on the validation battles:

- the reach closure per step: error in units, error per unit alive, R², share of steps with Blue
  scattered, mean formation depth and width;
- rollouts of `model_derived`: health error at 4 s in absolute terms and relative to the starting
  total health, and margin R² at 2 and 4 s.

In [ ]:
SIZE_BINS = [0, 14, 18, 22, 40]
SIZE_LABELS = ["11-14", "15-18", "19-22", "23-33"]
vv = val[both].copy()
vv["size"] = pd.cut(vv["n_blue"] + vv["n_red"], SIZE_BINS, labels=SIZE_LABELS)
from sindy_wm.models.intent_model import SHAPE, STATE, team_shares  # noqa: E402

sb, sr = team_shares(vv[STATE].to_numpy(), vv[SHAPE].to_numpy(), current.P)
vv["clo_blue"], vv["clo_red"] = vv["alive_blue"] * sb, vv["alive_red"] * sr
vv["scattered"] = (vv["depth_blue"] > 1.5) | (vv["width_blue"] > 3)
per_size = []
for s, g in vv.groupby("size", observed=True):
    row = {"units (both sides)": s, "battles": g.index.get_level_values(0).nunique()}
    for side in ("blue", "red"):
        e = 10 * (g[f"reach_{side}"] - g[f"clo_{side}"])
        row[f"closure |error| {side} (units)"] = e.abs().mean()
        row[f"closure |error| {side} per unit alive"] = (e.abs() / (10 * g[f"alive_{side}"])).mean()
        row[f"closure R² {side}"] = r2(g[f"reach_{side}"], g[f"clo_{side}"])
    row["Blue scattered (% of steps)"] = 100 * g["scattered"].mean()
    row["depth Blue / Red"] = f"{g['depth_blue'].mean():.2f} / {g['depth_red'].mean():.2f}"
    row["width Blue / Red"] = f"{g['width_blue'].mean():.2f} / {g['width_red'].mean():.2f}"
    per_size.append(row)
display(pd.DataFrame(per_size).set_index("units (both sides)").T)

Yd = lab.simulate(lab.base_params(), Bv)
ep_size = val.groupby(level=0)[["n_blue", "n_red"]].first().sum(axis=1)
size_of_start = pd.cut(ep_size.reindex(Bv["ep"]).to_numpy(), SIZE_BINS, labels=SIZE_LABELS)
roll_size = []
for s in SIZE_LABELS:
    m = np.asarray(size_of_start == s)
    sub = {k: (v[m] if isinstance(v, np.ndarray) and len(v) == len(m) else v) for k, v in Bv.items()}
    met = lab.metrics(Yd[m], sub, horizons=(2, 4))
    n4 = round(4 / DT)
    ok = ~np.isnan(sub["Y"][:, n4, 0])
    start_total = sub["X0"][ok, 0] + sub["X0"][ok, 1]
    rel = (np.abs(Yd[m][ok, n4, :2] - sub["Y"][ok, n4, :2]).sum(1) / start_total).mean()
    roll_size.append(
        {
            "units (both sides)": s,
            "start points": int(m.sum()),
            "health |error| 4 s": met["health 4s"],
            "health error 4 s / starting total health (%)": 100 * rel,
            "margin R² 2 s": met["margin R2 2s"],
            "margin R² 4 s": met["margin R2 4s"],
            "alive |error| 4 s (units)": met["alive 4s"],
        }
    )
pd.DataFrame(roll_size).set_index("units (both sides)").round(3)

## Questions (for you)

1. Section 1-2: health, alive and reach are piecewise constant. What does "the derivative of
   team health" mean physically in this game, and at what time scale does it exist?
2. Section 3: is the sawtooth noise or a rhythm? What would a synchronised volley do to a model
   that assumes a smooth damage rate, and when in the battle does it matter?
3. Section 4: the model explains the smoothed rate much better than the raw rate. Is the
   remaining raw-rate variance something any team-level ODE could ever explain?
4. Section 5: which numbers stay put across derivative estimates, and which move? Does the term
   *selection* change? What does that say about SINDy's viability here, and about which numbers
   in your model you can report as robust?
5. Section 6: the effective cooldown is 0.625 s, not 0.61 s. Which of your "matches the game
   rule" numbers (e.g. 0.219 in sindyc_06 test 1) need correcting?
6. Section 7: does removing the rhythm exactly change the fitted equations or the predictions?
7. Section 8: cooldown explains the per-step sawtooth almost perfectly. Why does it still not
   help much over 1-8 s? What would a "volley phase" state cost in readability, and what would it
   buy?
8. Section 9: absolute errors grow with battle size, relative ones don't (or shrink). Which is
   the right measure for "does the team-level description get worse for large battles"? And
   does that support or contradict your prediction that the closure gets worse as forces spread?